In [0]:
# The plan:
# Read current HVAC state from Silver
# Apply rules
# Decide a recommended setpoint
# Save the recommendation into a table
# Later, use that recommendation in the simulator

# For now, we are not yet sending commands to real equipment.
# We are only simulating the control decision

# We will create:

# a control recommendation table
# a SQL rule logic
# a query to generate one recommendation per time interval
# a table to store the latest control decisions

# will base the rule on:

# return temperature
# flow
# power
# delta temperature

# Rule A — low load
# return temp is low
# flow is low
# power is low

# Then:
# raise chilled water supply setpoint slightly
# because system does not need to work so hard

# Rule B — high load
# return temp is high
# flow is high
# power is high

# Then:
# lower chilled water supply setpoint slightly
# because system needs more cooling

# Rule C — normal load
# If neither low nor high:
# keep setpoint unchanged

In [0]:
%sql
-- # Create control schema
CREATE SCHEMA IF NOT EXISTS hvacapp_dev3.serving;

-- # -- Create control recommendation table
CREATE OR REPLACE TABLE hvacapp_dev3.serving.control_recommendations (
  recommendation_ts TIMESTAMP,
  event_ts TIMESTAMP,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  current_supply_temp_c DOUBLE,
  return_temp_c DOUBLE,
  delta_temp_c DOUBLE,
  flow_lpm DOUBLE,
  actual_power_kw DOUBLE,
  current_setpoint_c DOUBLE,
  recommended_setpoint_c DOUBLE,
  control_mode STRING,
  rule_name STRING,
  reason STRING,
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
SELECT
  current_timestamp() AS recommendation_ts,
  event_ts,
  site_id,
  building_id,
  equipment_id AS hvac_system_id,
  chw_supply_temp_c AS current_supply_temp_c,
  chw_return_temp_c AS return_temp_c,
  (chw_return_temp_c - chw_supply_temp_c) AS delta_temp_c,
  chw_flow_lpm AS flow_lpm,
  hvac_power_kw AS actual_power_kw,
  9.0 AS current_setpoint_c,

  CASE
    WHEN chw_return_temp_c < 11.5
         AND chw_flow_lpm < 760
         AND hvac_power_kw < 330
      THEN 9.5

    WHEN chw_return_temp_c > 12.5
         AND chw_flow_lpm > 800
         AND hvac_power_kw > 360
      THEN 8.0

    ELSE 9.0
  END AS recommended_setpoint_c,

  'RULE_BASED' AS control_mode,

  CASE
    WHEN chw_return_temp_c < 11.5
         AND chw_flow_lpm < 760
         AND hvac_power_kw < 330
      THEN 'LOW_LOAD_RULE'

    WHEN chw_return_temp_c > 12.5
         AND chw_flow_lpm > 800
         AND hvac_power_kw > 360
      THEN 'HIGH_LOAD_RULE'

    ELSE 'NORMAL_LOAD_RULE'
  END AS rule_name,

  CASE
    WHEN chw_return_temp_c < 11.5
         AND chw_flow_lpm < 760
         AND hvac_power_kw < 330
      THEN 'Low load detected, relax setpoint to reduce energy use'

    WHEN chw_return_temp_c > 12.5
         AND chw_flow_lpm > 800
         AND hvac_power_kw > 360
      THEN 'High load detected, tighten setpoint to maintain cooling'

    ELSE 'Load is normal, keep current setpoint'
  END AS reason,

  current_timestamp() AS created_ts

FROM hvacapp_dev3.refined.silver_telemetry_clean
ORDER BY event_ts DESC;

In [0]:
%sql
DELETE FROM hvacapp_dev3.serving.control_recommendations;

In [0]:
%sql
INSERT INTO hvacapp_dev3.serving.control_recommendations
SELECT
  current_timestamp() AS recommendation_ts,
  event_ts,
  site_id,
  building_id,
  equipment_id AS hvac_system_id,
  chw_supply_temp_c AS current_supply_temp_c,
  chw_return_temp_c AS return_temp_c,
  (chw_return_temp_c - chw_supply_temp_c) AS delta_temp_c,
  chw_flow_lpm AS flow_lpm,
  hvac_power_kw AS actual_power_kw,
  9.0 AS current_setpoint_c,

  CASE
    WHEN chw_return_temp_c < 11.5
         AND chw_flow_lpm < 760
         AND hvac_power_kw < 330
      THEN 9.5

    WHEN chw_return_temp_c > 12.5
         AND chw_flow_lpm > 800
         AND hvac_power_kw > 360
      THEN 8.0

    ELSE 9.0
  END AS recommended_setpoint_c,

  'RULE_BASED' AS control_mode,

  CASE
    WHEN chw_return_temp_c < 11.5
         AND chw_flow_lpm < 760
         AND hvac_power_kw < 330
      THEN 'LOW_LOAD_RULE'

    WHEN chw_return_temp_c > 12.5
         AND chw_flow_lpm > 800
         AND hvac_power_kw > 360
      THEN 'HIGH_LOAD_RULE'

    ELSE 'NORMAL_LOAD_RULE'
  END AS rule_name,

  CASE
    WHEN chw_return_temp_c < 11.5
         AND chw_flow_lpm < 760
         AND hvac_power_kw < 330
      THEN 'Low load detected, relax setpoint to reduce energy use'

    WHEN chw_return_temp_c > 12.5
         AND chw_flow_lpm > 800
         AND hvac_power_kw > 360
      THEN 'High load detected, tighten setpoint to maintain cooling'

    ELSE 'Load is normal, keep current setpoint'
  END AS reason,

  current_timestamp() AS created_ts

FROM hvacapp_dev3.refined.silver_telemetry_clean;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.serving.control_recommendations
ORDER BY event_ts DESC;

In [0]:
%sql
SELECT
  rule_name,
  COUNT(*) AS row_count
FROM hvacapp_dev3.serving.control_recommendations
GROUP BY rule_name
ORDER BY row_count DESC;

In [0]:
%sql
SELECT
  AVG(current_setpoint_c) AS avg_current_setpoint,
  AVG(recommended_setpoint_c) AS avg_recommended_setpoint
FROM hvacapp_dev3.serving.control_recommendations;

In [0]:
%sql
-- # we want the latest recommendation only.
-- # Create a view for latest recommendation by timestamp

CREATE OR REPLACE VIEW hvacapp_dev3.serving.latest_control_recommendation AS
SELECT *
FROM hvacapp_dev3.serving.control_recommendations
ORDER BY event_ts DESC
LIMIT 1;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.serving.latest_control_recommendation;

In [0]:
%sql
-- # The previous view gives only one latest row overall.
-- # A better version is latest per system.

CREATE OR REPLACE VIEW hvacapp_dev3.serving.latest_control_recommendation_per_system AS
SELECT
  recommendation_ts,
  event_ts,
  site_id,
  building_id,
  hvac_system_id,
  current_supply_temp_c,
  return_temp_c,
  delta_temp_c,
  flow_lpm,
  actual_power_kw,
  current_setpoint_c,
  recommended_setpoint_c,
  control_mode,
  rule_name,
  reason,
  created_ts
FROM (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY hvac_system_id
           ORDER BY event_ts DESC, recommendation_ts DESC
         ) AS rn
  FROM hvacapp_dev3.serving.control_recommendations
) t
WHERE rn = 1;


In [0]:
%sql
SELECT *
FROM hvacapp_dev3.serving.latest_control_recommendation_per_system;

In [0]:
%sql
CREATE OR REPLACE VIEW hvacapp_dev3.serving.control_recommendations_with_flag AS
SELECT *,
  CASE
    WHEN recommended_setpoint_c <> current_setpoint_c THEN 'AI_ON'
    ELSE 'AI_STANDBY'
  END AS ai_control_flag
FROM hvacapp_dev3.serving.control_recommendations;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.serving.control_recommendations_with_flag
ORDER BY event_ts DESC;

In [0]:
%sql
SELECT
  ai_control_flag,
  COUNT(*) AS row_count
FROM hvacapp_dev3.serving.control_recommendations_with_flag
GROUP BY ai_control_flag;

In [0]:
%sql
SELECT
  rule_name,
  ROUND(AVG(actual_power_kw), 2) AS avg_power_kw,
  ROUND(AVG(recommended_setpoint_c), 2) AS avg_recommended_setpoint_c
FROM hvacapp_dev3.serving.control_recommendations
GROUP BY rule_name;